# Modul 05: Bild- und Signaldaten vorbereiten

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Bilddaten vorbereiten, Signale vorbereiten  
    **Erwarteter Schwierigkeitsgrad:** Leicht fortgeschritten  
    **Orientierungszeit:** etwa 100 bis 140 Minuten

    ## Überblick

    Sie erzeugen vollständig lokale Bild- und Signaldaten, untersuchen deren Formen und Wertebereiche und bauen wiederverwendbare Vorverarbeitungsschritte für Pixel, Zeitfenster, Statistik- und Frequenzmerkmale.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_05A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_05B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Kleine Bilder mit Pillow, OpenCV und NumPy laden, konvertieren und darstellen.
- Farbräume, Pixelwerte, Histogramme und grundlegende Bildoperationen untersuchen.
- Eine einheitliche Vorverarbeitung für kleine Bildsammlungen erstellen.
- Einfache Signale mit Zeitachse, Trend und Rauschen erzeugen und visualisieren.
- Fenster-, Änderungs- und Frequenzmerkmale aus Messreihen berechnen.
- Eine tabellarische Merkmalstabelle aus Signalabschnitten erstellen.

    ## Bewertete Fähigkeiten

    - RGB/BGR, Graustufen, Alpha, Normalisierung und Bildformen
- Zuschneiden, Skalieren, Glätten und Kantenerkennung
- Abtastrate, Zeitachse, gleitende Fenster und lokale Kennzahlen
- FFT, dominante Frequenz und Merkmalstabellen

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
from PIL import Image
from io import StringIO

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ein synthetisches RGB-Bild vermeidet externe Dateien und Downloads.
height, width = 120, 160
synthetic_rgb = np.zeros((height, width, 3), dtype=np.uint8)
synthetic_rgb[:, :, 0] = np.linspace(20, 220, width, dtype=np.uint8)
synthetic_rgb[:, :, 1] = np.linspace(220, 40, height, dtype=np.uint8)[:, None]
synthetic_rgb[35:90, 50:115, 2] = 255
cv2.circle(synthetic_rgb, center=(35, 35), radius=18, color=(255, 255, 40), thickness=-1)

# Ein Signal mit zwei Frequenzanteilen, Trend und reproduzierbarem Rauschen.
sampling_rate_hz = 100
duration_seconds = 4
time_axis = np.arange(0, duration_seconds, 1 / sampling_rate_hz)
base_signal = (
    1.2 * np.sin(2 * np.pi * 5 * time_axis)
    + 0.45 * np.sin(2 * np.pi * 14 * time_axis)
    + 0.08 * time_axis
)
noisy_signal = base_signal + rng.normal(0, 0.18, size=time_axis.size)

print("Bildform:", synthetic_rgb.shape, synthetic_rgb.dtype)
print("Signalpunkte:", noisy_signal.size, "Abtastrate:", sampling_rate_hz, "Hz")

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Bildformate, Kanäle und Pixelwerte untersuchen

    Arbeiten Sie mit `synthetic_rgb`.

1. Erzeugen Sie ein Pillow-Bild und geben Sie Modus und Größe aus.
2. Konvertieren Sie RGB nach BGR und zurück. Prüfen Sie, ob das zurückkonvertierte Bild identisch ist.
3. Erzeugen Sie ein Graustufenbild.
4. Ergänzen Sie einen Alpha-Kanal mit einem horizontalen Transparenzverlauf.
5. Zeigen Sie RGB-, Graustufen- und RGBA-Bild in getrennten Abbildungen.

> **Hinweis:** Prüfen Sie Form, Datentyp und Wertebereich nach jeder Konvertierung.

In [ ]:
image_rgb = synthetic_rgb.copy()

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Bildformate, Kanäle und Pixelwerte untersuchen
#
# Ziel dieser Codezelle:
# Arbeiten Sie mit syntheticrgb. 1. Erzeugen Sie ein Pillow-Bild und geben Sie Modus
# und Größe aus. 2. Konvertieren Sie RGB nach BGR und zurück. Prüfen Sie, ob das
# zurückkonvertierte Bild identisch ist. 3. Erzeugen Sie...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

image_rgb = synthetic_rgb.copy()

# Pillow erwartet bei einem uint8-Array mit drei Kanälen standardmäßig RGB.
pil_image = Image.fromarray(image_rgb, mode="RGB")
print("Pillow-Modus:", pil_image.mode)
print("Pillow-Größe (Breite, Höhe):", pil_image.size)

# OpenCV verwendet bei vielen Funktionen die Reihenfolge BGR. Die
# explizite Konvertierung verhindert vertauschte Rot- und Blaukanäle.
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
image_rgb_roundtrip = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print("RGB-Rückkonvertierung identisch:", np.array_equal(image_rgb, image_rgb_roundtrip))

# Graustufen fassen die Farbinformation zu einem Intensitätskanal zusammen.
image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# Der Alpha-Kanal läuft von vollständig transparent (0) bis vollständig
# deckend (255). Er muss dieselbe Höhe und Breite wie das Bild besitzen.
alpha_row = np.linspace(0, 255, width, dtype=np.uint8)
alpha_channel = np.tile(alpha_row, (height, 1))
image_rgba = np.dstack([image_rgb, alpha_channel])

print("RGB-Form:", image_rgb.shape)
print("Graustufenform:", image_gray.shape)
print("RGBA-Form:", image_rgba.shape)

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image_rgb)
ax.set_title("Synthetisches RGB-Bild")
ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image_gray, cmap="gray")
ax.set_title("Graustufenbild")
ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image_rgba)
ax.set_title("RGBA mit Transparenzverlauf")
ax.axis("off")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

NumPy beschreibt Bilder meist als `(Höhe, Breite, Kanäle)`, Pillow gibt die Größe dagegen als `(Breite, Höhe)` aus. RGB und BGR enthalten dieselben Werte in anderer Kanalreihenfolge. Ein Graustufenbild besitzt nur zwei Dimensionen. Der Alpha-Kanal verändert nicht die Farbinformation, sondern steuert die Deckkraft.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Normalisieren, zuschneiden, skalieren und Kanten finden

    Erstellen Sie eine Funktion `prepare_image(image_rgb, output_size=(64, 64))`, die:

1. den mittleren quadratischen Bereich des Bildes ausschneidet,
2. auf die Zielgröße skaliert,
3. Pixelwerte in `float32` und den Bereich `[0, 1]` überführt,
4. zusätzlich ein geglättetes Graustufenbild und eine Canny-Kantenkarte erzeugt,
5. Formen, Datentypen und Wertebereiche prüft.

Zeigen Sie die drei Ergebnisse in getrennten Abbildungen.

> **Hinweis:** OpenCV verwendet bei Größenangaben häufig die Reihenfolge Breite, Höhe.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Normalisieren, zuschneiden, skalieren und Kanten finden
#
# Ziel dieser Codezelle:
# Erstellen Sie eine Funktion prepareimage(imagergb, outputsize=(64, 64)), die: 1.
# den mittleren quadratischen Bereich des Bildes ausschneidet, 2. auf die Zielgröße
# skaliert, 3. Pixelwerte in float32 und den Bereich [0,...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def prepare_image(
    image_rgb: np.ndarray,
    output_size: tuple[int, int] = (64, 64),
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Erzeugt normalisiertes RGB, geglättetes Grau und Kantenkarte.

    # Grundprüfungen machen Annahmen über die Eingabe ausdrücklich.
    if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
        raise ValueError("Erwartet wird ein RGB-Bild mit drei Kanälen.")
    if image_rgb.dtype != np.uint8:
        raise TypeError("Die Eingabe soll uint8-Pixel im Bereich 0..255 enthalten.")

    image_height, image_width, _ = image_rgb.shape
    side = min(image_height, image_width)

    # Die Startpositionen zentrieren den quadratischen Ausschnitt.
    start_y = (image_height - side) // 2
    start_x = (image_width - side) // 2
    square_crop = image_rgb[
        start_y : start_y + side,
        start_x : start_x + side,
    ]

    # OpenCV erwartet die Zielgröße als (Breite, Höhe).
    resized_rgb = cv2.resize(
        square_crop,
        dsize=output_size,
        interpolation=cv2.INTER_AREA,
    )

    # Die Division nach astype vermeidet ganzzahlige Rundung.
    normalized_rgb = resized_rgb.astype(np.float32) / 255.0

    # Für Glättung und Canny werden weiterhin uint8-Werte verwendet.
    gray = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2GRAY)
    blurred_gray = cv2.GaussianBlur(gray, ksize=(5, 5), sigmaX=0)
    edges = cv2.Canny(blurred_gray, threshold1=50, threshold2=130)

    # Kontrollprüfungen verhindern stillschweigende Form- oder Bereichsfehler.
    expected_shape = (output_size[1], output_size[0], 3)
    if normalized_rgb.shape != expected_shape:
        raise ValueError(f"Unerwartete RGB-Form: {normalized_rgb.shape}")
    if not (0.0 <= normalized_rgb.min() <= normalized_rgb.max() <= 1.0):
        raise ValueError("Normalisierte Pixel liegen nicht vollständig in [0, 1].")

    return normalized_rgb, blurred_gray, edges

normalized_rgb, blurred_gray, edge_map = prepare_image(synthetic_rgb)

print("Normalisiert:", normalized_rgb.shape, normalized_rgb.dtype, normalized_rgb.min(), normalized_rgb.max())
print("Geglättet:", blurred_gray.shape, blurred_gray.dtype)
print("Kanten:", edge_map.shape, edge_map.dtype, "Kantenpixel:", int((edge_map > 0).sum()))

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(normalized_rgb)
ax.set_title("Zugeschnitten und normalisiert")
ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(blurred_gray, cmap="gray")
ax.set_title("Geglättetes Graustufenbild")
ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(edge_map, cmap="gray")
ax.set_title("Canny-Kantenkarte")
ax.axis("off")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 2

Zuschneiden und Skalieren vereinheitlichen die räumliche Form. Normalisierung ändert Wertebereich und Datentyp, nicht den Bildinhalt. Glättung reduziert kleine Schwankungen, bevor Canny starke Intensitätsänderungen als Kanten markiert. Schwellenwerte beeinflussen die Anzahl gefundener Kanten und sollten für einen Datensatz einheitlich dokumentiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Zeitachse, Abtastung und Messreihe prüfen

    Erzeugen Sie aus `time_axis` und `noisy_signal` eine Tabelle. Speichern Sie sie in einen CSV-Text im Arbeitsspeicher und laden Sie sie mit `StringIO` erneut.

1. Prüfen Sie, ob Zeitabstände konstant sind.
2. Schätzen Sie die Abtastrate aus dem Median der Zeitabstände.
3. Fügen Sie absichtlich zwei Fehlwerte und einen starken Ausreißer ein.
4. Markieren Sie die betroffenen Zeilen.
5. Zeichnen Sie das ursprüngliche und das fehlerhafte Signal in getrennten Abbildungen.

> **Hinweis:** Leiten Sie die Abtastrate aus der Zeitachse ab, statt sie nur anzunehmen.

In [ ]:
signal_table = pd.DataFrame({"time_s": time_axis, "amplitude": noisy_signal})

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Zeitachse, Abtastung und Messreihe prüfen
#
# Ziel dieser Codezelle:
# Erzeugen Sie aus timeaxis und noisysignal eine Tabelle. Speichern Sie sie in einen
# CSV-Text im Arbeitsspeicher und laden Sie sie mit StringIO erneut. 1. Prüfen Sie,
# ob Zeitabstände konstant sind. 2. Schätzen Sie die A...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

signal_table = pd.DataFrame(
    {"time_s": time_axis, "amplitude": noisy_signal}
)

# Das Schreiben und erneute Lesen simuliert eine lokale CSV-Quelle,
# ohne eine externe Datei oder Internetzugriff zu benötigen.
csv_text = signal_table.to_csv(index=False)
loaded_signal = pd.read_csv(StringIO(csv_text))

time_differences = np.diff(loaded_signal["time_s"].to_numpy())
constant_spacing = np.allclose(
    time_differences,
    np.median(time_differences),
    rtol=1e-6,
    atol=1e-9,
)
estimated_sampling_rate = 1.0 / np.median(time_differences)

# Eine Kopie erhält die saubere Referenz für den visuellen Vergleich.
corrupted_signal = loaded_signal.copy()
corrupted_signal.loc[[80, 210], "amplitude"] = np.nan
corrupted_signal.loc[310, "amplitude"] = 5.5

# Der Ausreißer wird hier mit einem robusten Abstand vom Median markiert.
median_amplitude = corrupted_signal["amplitude"].median()
median_absolute_deviation = (
    corrupted_signal["amplitude"] - median_amplitude
).abs().median()
robust_distance = (
    (corrupted_signal["amplitude"] - median_amplitude).abs()
    / max(median_absolute_deviation, 1e-12)
)

corrupted_signal["is_missing"] = corrupted_signal["amplitude"].isna()
corrupted_signal["is_large_outlier"] = robust_distance > 8

print("Konstante Zeitabstände:", constant_spacing)
print("Geschätzte Abtastrate:", round(estimated_sampling_rate, 3), "Hz")
print("Markierte Zeilen:")
print(
    corrupted_signal.loc[
        corrupted_signal[["is_missing", "is_large_outlier"]].any(axis=1)
    ].to_string(index=False)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loaded_signal["time_s"], loaded_signal["amplitude"], linewidth=1)
ax.set_title("Saubere geladene Messreihe")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Amplitude")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(corrupted_signal["time_s"], corrupted_signal["amplitude"], linewidth=1)
ax.scatter(
    corrupted_signal.loc[corrupted_signal["is_large_outlier"], "time_s"],
    corrupted_signal.loc[corrupted_signal["is_large_outlier"], "amplitude"],
    marker="x",
    s=90,
    label="markierter Ausreißer",
)
ax.set_title("Messreihe mit Fehlwerten und Ausreißer")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Amplitude")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 3

Eine konstante Abtastung ist Voraussetzung für eine einfache Interpretation der FFT. Fehlwerte erzeugen Lücken und müssen vor Fenster- oder Frequenzberechnungen behandelt werden. Der robuste Ausreißerhinweis nutzt Median und mediane absolute Abweichung, ist aber ebenfalls nur ein Prüfhinweis. Ein starker Ausschlag kann Messfehler oder ein reales Ereignis sein.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Fenster-, Änderungs- und Frequenzmerkmale berechnen

    Schreiben Sie eine Funktion `signal_window_features(values, sampling_rate, window_size, step_size)`, die über gleitende Fenster folgende Merkmale erzeugt:

- Start- und Endindex,
- Mittelwert,
- Standardabweichung,
- RMS-Wert,
- maximaler absoluter erster Unterschied,
- dominante positive Frequenz ohne Gleichanteil.

Wenden Sie die Funktion mit Fenstergröße 100 und Schrittweite 50 auf `noisy_signal` an. Visualisieren Sie die dominante Frequenz je Fenster.

> **Hinweis:** Entfernen Sie den Fenstermittelwert, bevor Sie die stärkste positive Frequenz suchen.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Fenster-, Änderungs- und Frequenzmerkmale berechnen
#
# Ziel dieser Codezelle:
# Schreiben Sie eine Funktion signalwindowfeatures(values, samplingrate, windowsize,
# stepsize), die über gleitende Fenster folgende Merkmale erzeugt: - Start- und
# Endindex, - Mittelwert, - Standardabweichung, - RMS-Wert...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def signal_window_features(
    values: np.ndarray,
    sampling_rate: float,
    window_size: int,
    step_size: int,
) -> pd.DataFrame:
    # Berechnet lokale Statistik- und Frequenzmerkmale.

    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1:
        raise ValueError("Das Signal muss eindimensional sein.")
    if window_size <= 1 or step_size <= 0:
        raise ValueError("Fenster- und Schrittgröße müssen positiv sein.")

    feature_rows = []

    # Das letzte Fenster wird nur verwendet, wenn es vollständig ist.
    for start in range(0, len(values) - window_size + 1, step_size):
        end = start + window_size
        window = values[start:end]

        # Zeitbereichsmerkmale fassen Niveau, Streuung, Energie und
        # stärkste lokale Änderung zusammen.
        mean_value = float(np.mean(window))
        std_value = float(np.std(window, ddof=0))
        rms_value = float(np.sqrt(np.mean(window**2)))
        max_abs_difference = float(np.max(np.abs(np.diff(window))))

        # Vor der FFT wird der Mittelwert entfernt. Dadurch dominiert
        # der Gleichanteil bei 0 Hz nicht die Frequenzsuche.
        centered_window = window - mean_value
        spectrum = np.abs(np.fft.rfft(centered_window))
        frequencies = np.fft.rfftfreq(
            window_size,
            d=1.0 / sampling_rate,
        )
        spectrum[0] = 0.0
        dominant_frequency = float(
            frequencies[np.argmax(spectrum)]
        )

        feature_rows.append(
            {
                "start_index": start,
                "end_index": end,
                "mean": mean_value,
                "std": std_value,
                "rms": rms_value,
                "max_abs_difference": max_abs_difference,
                "dominant_frequency_hz": dominant_frequency,
            }
        )

    return pd.DataFrame(feature_rows)

window_features = signal_window_features(
    noisy_signal,
    sampling_rate=sampling_rate_hz,
    window_size=100,
    step_size=50,
)

print(window_features.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    window_features["start_index"] / sampling_rate_hz,
    window_features["dominant_frequency_hz"],
    marker="o",
)
ax.set_title("Dominante Frequenz je gleitendem Fenster")
ax.set_xlabel("Fensterstart in Sekunden")
ax.set_ylabel("Dominante Frequenz in Hz")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

Überlappende Fenster liefern eine zeitlich feinere Folge von Merkmalen als nicht überlappende Fenster, erzeugen aber stärker abhängige Beobachtungen. Die dominante Frequenz liegt in diesem Signal meist nahe 5 Hz, weil dieser Anteil die größere Amplitude besitzt. Fensterlänge und Abtastrate bestimmen die erreichbare Frequenzauflösung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Kleine Bild- und Signal-Batches vorbereiten

    Erstellen Sie zwei wiederverwendbare Ausgaben.

**Bildteil:** Erzeugen Sie aus drei Varianten von `synthetic_rgb` einen Batch normalisierter Bilder der Form `(3, 64, 64, 3)`. Verwenden Sie Original, horizontal gespiegeltes Bild und eine dunklere Variante.

**Signalteil:** Erzeugen Sie sechs Signale, drei mit ungefähr 5 Hz und drei mit ungefähr 12 Hz. Berechnen Sie pro vollständigem Signal dieselben Statistik- und Frequenzmerkmale und erstellen Sie eine Merkmalstabelle mit Zielspalte `signal_class`.

Prüfen Sie Formen, Datentypen, Wertebereiche und Klassenverteilung.

> **Hinweis:** Die Batchdimension zählt Beispiele, nicht Bildkanäle oder Zeitpunkte.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Kleine Bild- und Signal-Batches vorbereiten
#
# Ziel dieser Codezelle:
# Erstellen Sie zwei wiederverwendbare Ausgaben. Bildteil: Erzeugen Sie aus drei
# Varianten von syntheticrgb einen Batch normalisierter Bilder der Form (3, 64, 64,
# 3). Verwenden Sie Original, horizontal gespiegeltes Bild...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# -----------------------------
# Bild-Batch
# -----------------------------
image_variants = [
    synthetic_rgb,
    np.fliplr(synthetic_rgb),
    np.clip(synthetic_rgb.astype(np.float32) * 0.60, 0, 255).astype(np.uint8),
]

prepared_images = []
for image in image_variants:
    # Der mittlere quadratische Bereich wird ausgeschnitten und auf
    # dieselbe räumliche Form gebracht.
    h, w, _ = image.shape
    side = min(h, w)
    y0 = (h - side) // 2
    x0 = (w - side) // 2
    crop = image[y0 : y0 + side, x0 : x0 + side]
    resized = cv2.resize(crop, (64, 64), interpolation=cv2.INTER_AREA)
    prepared_images.append(resized.astype(np.float32) / 255.0)

image_batch = np.stack(prepared_images, axis=0)

# -----------------------------
# Signal-Merkmalstabelle
# -----------------------------
signal_rows = []
local_time = np.arange(0, 2, 1 / sampling_rate_hz)

for signal_class, frequency in [("niedrig", 5), ("hoch", 12)]:
    for replicate in range(3):
        # Jede Wiederholung erhält leicht anderes Rauschen und Phase.
        phase = rng.uniform(0, 2 * np.pi)
        signal_values = (
            np.sin(2 * np.pi * frequency * local_time + phase)
            + rng.normal(0, 0.15, size=local_time.size)
        )

        centered = signal_values - signal_values.mean()
        spectrum = np.abs(np.fft.rfft(centered))
        frequencies = np.fft.rfftfreq(
            signal_values.size,
            d=1.0 / sampling_rate_hz,
        )
        spectrum[0] = 0.0

        signal_rows.append(
            {
                "signal_class": signal_class,
                "replicate": replicate,
                "mean": float(signal_values.mean()),
                "std": float(signal_values.std(ddof=0)),
                "rms": float(np.sqrt(np.mean(signal_values**2))),
                "max_abs_difference": float(np.max(np.abs(np.diff(signal_values)))),
                "dominant_frequency_hz": float(frequencies[np.argmax(spectrum)]),
            }
        )

signal_feature_table = pd.DataFrame(signal_rows)

# Kontrollprüfungen machen die erwarteten Lernformen sichtbar.
assert image_batch.shape == (3, 64, 64, 3)
assert image_batch.dtype == np.float32
assert 0.0 <= image_batch.min() <= image_batch.max() <= 1.0
assert signal_feature_table["signal_class"].value_counts().to_dict() == {
    "niedrig": 3,
    "hoch": 3,
}

print("Bild-Batch:", image_batch.shape, image_batch.dtype)
print("Bildbereich:", float(image_batch.min()), "bis", float(image_batch.max()))
print("\nSignalmerkmale:")
print(signal_feature_table.round(3).to_string(index=False))
print("\nKlassenverteilung:")
print(signal_feature_table["signal_class"].value_counts().to_string())

### Reflexion zu Aufgabe 5

Der Bild-Batch besitzt eine explizite Beobachtungsdimension vor Höhe, Breite und Kanälen. Alle Bilder müssen dieselbe Form und denselben Wertebereich haben. Die Signalmerkmalstabelle übersetzt jede vollständige Messreihe in eine Tabellenzeile. Die dominante Frequenz sollte die beiden Klassen deutlich trennen, während Mittelwert oder Standardabweichung allein weniger aussagekräftig sein können. Bei späteren zeitlichen Modellen dürfen überlappende Fenster derselben Messreihe nicht zufällig auf Train und Test verteilt werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.